In [23]:

import json
import urllib3
from urllib.parse import urljoin
import certifi
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import time 
import re

import pandas as pd

In [3]:
http = urllib3.PoolManager(
    ca_certs=certifi.where()
)

url= "https://www.thegradcafe.com"
response = http.request("GET", url)
print(response.status)

html = response.data.decode("utf-8")
print(html[:1000])

soup = BeautifulSoup(html, "html.parser")
text = soup.get_text()
spaceless_text = text.replace("\n\n", "")
print(spaceless_text)

# get the Page title
print(soup.title.string) 

200
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">

    <title inertia>The GradCafe</title>

    <!-- Fonts -->
    

    <!-- Third-party tokens -->
            <meta name="am-api-token" content="DbyJ3SXLe" />
    
    <!-- Scripts -->
    <script type="text/javascript">const Ziggy={"url":"https:\/\/www.thegradcafe.com","port":null,"defaults":{},"routes":{"passport.token":{"uri":"oauth2\/token","methods":["POST"]},"passport.authorizations.authorize":{"uri":"oauth2\/authorize","methods":["GET","HEAD"]},"passport.device":{"uri":"oauth2\/device","methods":["GET","HEAD"]},"passport.device.code":{"uri":"oauth2\/device\/code","methods":["POST"]},"passport.token.refresh":{"uri":"oauth2\/token\/refresh","methods":["POST"]},"passport.authorizations.approve":{"uri":"oauth2\/authorize","methods":["POST"]},"passport.authorizations.deny":{"uri":"oauth2\/authorize","methods":["DELETE"]},"passport.device.au

In [4]:
#------------ use beautiful soup to find all hyperlinks on this page -----------#

for link in soup.find_all("a"):   #Find all HTML <a> (anchor) tags in the page."
    text = link.get_text(strip=True)  #get all the text
    print(text)

#------------- use beautiful soup to find hyperlink for admissions ------------#
# for link in soup.find_all("a"):   #Find all HTML <a> (anchor) tags in the page."
#     text = link.get_text(strip=True)  
#     if "Admissions" in text:
#         print("Text:", text)
#         results = link.get("href")  #get the URL from the href attribute of the <a> tag
#         print("URL :", results)

url_admissions = None

for link in soup.find_all("a"): #Find all HTML <a> (anchor) tags in the page."
    text = link.get_text(strip=True)
    href = link.get("href")  #Get the value of the href attribute from the HTML tag. 
    if text == "Admissions" and "blog" not in href:
        url_admissions = href
        break

print(url_admissions)
# https://www.thegradcafe.com/survey


Logo
Admissions
Submit Your Results
Blog
Forum
Sign In
Submit Your Results Now
Planning
How to Prepare for Grad School: Top Tips
Chriselle Sy
Admissions
The Grad School Conundrum: What Acceptance Really Means
The GradCafe Editor
Admissions
Don’t Go Through the Grad School Application Process Alone
Dr. Robert Johns
University of California, San DiegoComputer Science
ETH ZurichComputer Science
University of British ColumbiaComputer Science
Cornell UniversityApplied Statistics
University of WashingtonArchitecture
Columbia UniversityArtificial Intelligence
University of TorontoSocial Work
University of California, IrvineComputer Science
Stanford UniversityComputer Science
University of California, San DiegoElectrical And Computer Engineering
Harvard UniversityEconomics
University of California (UCLA)Chemistry
Massachusetts Institute of Technology (MIT)Physics
Georgia Institute of TechnologyComputer Science
University of Illinois Urbana-ChampaignPhysics
New York UniversitySociology
Universi

In [ ]:
#------------------------------ test selenium, scrape one page only-----------------------------#

# driver = webdriver.Chrome()

# #  load a web page by navigating to the provided URL
# driver.get(url_admissions)

# # Wait until at least one result row appears
# WebDriverWait(driver, 20).until(
#     EC.presence_of_element_located((By.TAG_NAME, "table"))
# )

# html_test = driver.page_source #extract the rendered html

# print("Page loaded")
# print(html_test[:500])

# # Close browser
# driver.quit()


# # check how much data has been downloaded
# print(len(html_test))

Page loaded
<html lang="en"><head><script async="" defer="" src="https://launchpad.privacymanager.io/latest/launchpad.bundle.js"></script><script type="text/javascript" src="https://socialcanvas-cdn.kargo.com/js/spotlight/latest/spotlight.min.js" fetchpriority="high"></script><script type="text/javascript" async="" src="https://cdn.id5-sync.com/api/1.0/id5PrebidModule.js"></script>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">

    <link rel="shortcut ic
218408


In [ ]:
#--------------------- scrape more pages, test how many  -------------------------#
driver = webdriver.Chrome()

records = []

try:
    for page in range(1, 6):  # Pages 1 through 5

        url = f"https://www.thegradcafe.com/survey?page={page}"
        print(f"\nScraping page {page}")

        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "table"))
        )

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        rows = soup.find_all("tr")

        print(f"Found {len(rows)} rows")

        for tr in rows:
            text = tr.get_text(" ", strip=True)

            if not text:
                continue

            records.append({
                "page": page,
                "raw_text": text
            })

        time.sleep(1)

finally:
    driver.quit()

print("\nTotal records scraped:", len(records))

# since each page has around 50 records, I need to scrape around 1000 pages to reach 50000 records


Scraping page 1
Found 56 rows

Scraping page 2
Found 49 rows

Scraping page 3
Found 53 rows

Scraping page 4
Found 51 rows

Scraping page 5
Found 50 rows

Total records scraped: 244


In [26]:
# # -------version 2---------------------------
# # start_time = time.time() 
# base_url = "https://www.thegradcafe.com"

# driver = webdriver.Chrome()
# raw_data = []

# try:
#     for page in range(1, 5):
#         url = f"https://www.thegradcafe.com/survey?page={page}"
#         print(f"\nScraping page {page}")

#         driver.get(url)

#         WebDriverWait(driver, 20).until(
#             EC.presence_of_element_located((By.TAG_NAME, "table"))
#         )

#         html = driver.page_source
#         soup = BeautifulSoup(html, "html.parser")

#         # rows = soup.find_all("tr")
#         # print(f"Rows found on page {page}: {len(rows)}")
        
#         table = soup.find("table")
#         if table is None:
#             print("No table found")
#             continue

#         headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
#         print("Headers found:", headers)

#         rows = table.find_all("tr")
#         print(f"Rows found on page {page}: {len(rows)}")

#         for tr in rows:
#             cells = tr.find_all("td")
#             if not cells:
#                 continue
            
#             cell_texts = [cell.get_text(" ", strip=True) for cell in cells]
#             raw_text = tr.get_text(" ", strip=True)

#             cell_header_mapping = dict(zip(headers, cell_texts))
#             print("Cell header mapping:", cell_header_mapping)

#             # find hyperlinks in the row
#             link_tag = tr.find("a", href=True)
#             applicant_entry_url = urljoin(base_url, link_tag["href"]) if link_tag else None

#             raw_data.append({"page_number": page,
#                             "raw_text": raw_text,
#                             "cell_texts": cell_texts,
#                             "cell_header_mapping": cell_header_mapping,
#                             "applicant_entry_url": applicant_entry_url})

#         time.sleep(1.5)

# finally:
#     driver.quit()

# print("Total records:", len(raw_data))

# for i in raw_data[:3]:
#     print(i)

base_url= "https://www.thegradcafe.com"

driver = webdriver.Chrome()
raw_data = []

try:
    for page in range(1, 5):
        url = f"https://www.thegradcafe.com/survey?page={page}"
        print(f"\nScraping page {page}")

        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "table"))
        )

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        
        # find table
        table = soup.find("table")
        # find header
        headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
        print("Headers:", headers)

        current_record = None

        for tr in table.find_all("tr"):
            cells = tr.find_all("td")
            if not cells:
                continue
            
            # find raw text then quote with " "
            cell_texts = [cell.get_text(" ", strip=True) for cell in cells]

            # Main row: School, Program, Added On, Decision, Sort
            if len(cell_texts) == len(headers):
                link_tag = tr.find("a", href=True)
                applicant_entry_url = urljoin(base_url, link_tag["href"]) if link_tag else None

                cell_map = dict(zip(headers, cell_texts))

                current_record = {
                    "page_number": page,
                    "School": cell_map.get("School"),
                    "Program": cell_map.get("Program"),
                    "Added On": cell_map.get("Added On"),
                    "Decision": cell_map.get("Decision"),
                    "applicant_entry_url": applicant_entry_url,
                    "extra_info": None,   # placeholder
                    "comments": None,     # placeholder
                    "raw_texts": cell_texts
                }

                raw_data.append(current_record)

            # Other info/comment rows below the main row
            # else if means --- If this row is not a normal main row, but we already have a previous applicant record, 
            # then treat this row as extra information for that previous applicant.”

            elif current_record is not None:
                extra_info = " ".join(cell_texts)

                if current_record["extra_info"] is None:
                    current_record["extra_info"] = extra_info
                # If this applicant already has extra_info, then this next extra row is probably the comment.”
                else:  
                    current_record["comments"] = extra_info

        time.sleep(1.5)

finally:
    driver.quit()

print("Total records:", len(raw_data))



Scraping page 1
Headers: ['School', 'Program', 'Added On', 'Decision', 'Sort Newest Decision Date']

Scraping page 2
Headers: ['School', 'Program', 'Added On', 'Decision', 'Sort Newest Decision Date']

Scraping page 3
Headers: ['School', 'Program', 'Added On', 'Decision', 'Sort Newest Decision Date']

Scraping page 4
Headers: ['School', 'Program', 'Added On', 'Decision', 'Sort Newest Decision Date']
Total records: 80


In [19]:
# check data so far
for i in raw_data[:3]:
    print(i)

{'page_number': 1, 'School': 'San Jose State University', 'Program': 'Bioinformatics Masters', 'Added On': 'May 31, 2026', 'Decision': 'Rejected on May 31', 'applicant_entry_url': 'https://www.thegradcafe.com/result/1020289', 'extra_info': 'Rejected on May 31 Fall 2026 American GPA 3.20', 'comments': None, 'raw_texts_main_row': ['San Jose State University', 'Bioinformatics Masters', 'May 31, 2026', 'Rejected on May 31', 'Total comments']}
{'page_number': 1, 'School': 'Stockholm University', 'Program': 'Environmental Economics PhD', 'Added On': 'May 31, 2026', 'Decision': 'Accepted on May 31', 'applicant_entry_url': 'https://www.thegradcafe.com/result/1020288', 'extra_info': 'Accepted on May 31 Fall 2026 International', 'comments': None, 'raw_texts_main_row': ['Stockholm University', 'Environmental Economics PhD', 'May 31, 2026', 'Accepted on May 31', 'Total comments']}
{'page_number': 1, 'School': 'Meharry Medical College', 'Program': 'Global Health PhD', 'Added On': 'May 30, 2026', 'D

#### Clean data

In [27]:
def clean_space(value):
    """ 
    this function collapses multiple white spaces into one
    remove leading space and trailing space
    """
    if value is None:
        return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value if value else None


def extract_status(text):
    """ 
    this function detects the applicant status from the text, and return one of 
    the following categories
    """
    if text is not None: 
        text_lower = text.lower()

        if "not accepted" in text_lower:
            return None
        if "accepted" in text_lower:
            return "Accepted"
        if "rejected" in text_lower:
            return "Rejected"
        if "wait listed" in text_lower or "waitlisted" in text_lower:
            return "Waitlisted"
        if "interview" in text_lower:
            return "Interview"
    return None


def extract_decision_date(text):
    if text is None:
        return None

    match = re.search(
        r"\b(?:Accepted|Rejected|Interview|Wait listed|Waitlisted)\s+on\s+([A-Za-z]{3,9}\s+\d{1,2})",
        text,
        re.IGNORECASE)
    return match.group(1) if match else None


def extract_term(text):
    text = text or ""
    match = re.search(r"\b(Fall|Spring|Summer|Winter)\s+(20\d{2})\b", text, re.IGNORECASE)
    if match:
        return f"{match.group(1).title()} {match.group(2)}"
    return None


def extract_student_type(text):
    if text is None:
        return None
    
    text_lower = text.lower()

    if "international" in text_lower:
        return "International"
    if "american" in text_lower or "domestic" in text_lower:
        return "American"
    if "other" in text_lower:
        return "Other"
    return None


def extract_gpa(text):
    text = text or ""
    match = re.search(r"\bGPA\s*[: ]\s*([0-4](?:\.\d{1,3})?)\b", text, re.IGNORECASE)
    return match.group(1) if match else None


def extract_gre_score(text):
    text = text or ""
    match = re.search(r"\bGRE\s+(\d{2,3})\b", text, re.IGNORECASE)
    return match.group(1) if match else None


def extract_gre_v(text):
    text = text or ""
    match = re.search(r"\bGRE V\s+(\d{2,3})\b", text, re.IGNORECASE)
    return match.group(1) if match else None


def extract_gre_aw(text):
    text = text or ""
    match = re.search(r"\bGRE AW\s+(\d(?:\.\d{1,2})?)\b", text, re.IGNORECASE)
    return match.group(1) if match else None


def extract_degree(program):
    program = program or ""

    if "phd" in program.lower():
        return "PhD"
    if "masters" in program.lower() or "master" in program.lower():
        return "Masters"

    return None


cleaned_records = []

# step1-- clean space
for record in raw_data:
    extra_info = clean_space(record["extra_info"])
    comments = clean_space(record["comments"])
    decision_clean = clean_space(record["Decision"])

    combined_text = " ".join([
        text for text in [decision_clean, extra_info, comments]
        if text
    ])
    # two possible places to extract status
    status = extract_status(decision_clean) or extract_status(extra_info)
    decision_date = extract_decision_date(decision_clean) or extract_decision_date(extra_info)

    cleaned_records.append({
        "Program Name": clean_space(record["Program"]),
        "University": clean_space(record["School"]),
        "Comments": comments,
        "Date of Information Added to Grad Cafe": clean_space(record["Added On"]),
        "URL link to applicant entry": record["applicant_entry_url"],
        "Applicant Status": status,
        "Accepted: Acceptance Date": decision_date if status == "Accepted" else None,
        "Rejected: Rejection Date": decision_date if status == "Rejected" else None,
        "Semester and Year of Program Start": extract_term(combined_text),
        "International / American Student": extract_student_type(combined_text),
        "GRE Score": extract_gre_score(combined_text),
        "GRE V Score": extract_gre_v(combined_text),
        "Masters or PhD": extract_degree(record["Program"]),
        "GPA": extract_gpa(combined_text),
        "GRE AW": extract_gre_aw(combined_text),
        "extra_info": extra_info,
        "raw_texts": record["raw_texts"],
        "page_number": record["page_number"]
    })

df_cleaned = pd.DataFrame(cleaned_records)
df_cleaned.head()

,Program Name,University,Comments,Date of Information Added to Grad Cafe,URL link to applicant entry,Applicant Status,Accepted: Acceptance Date,Rejected: Rejection Date,Semester and Year of Program Start,International / American Student,GRE Score,GRE V Score,Masters or PhD,GPA,GRE AW,extra_info,raw_texts,page_number
0,Bioinformatics Masters,San Jose State University,NaN,"May 31, 2026",https://www.thegradcafe.com/result/1020289,Rejected,NaN,May 31,Fall 2026,American,NaN,NaN,Masters,3.20,NaN,Rejected on May 31 Fall 2026 American GPA 3.20,"[San Jose State University, Bioinformatics Mas...",1
1,Environmental Economics PhD,Stockholm University,NaN,"May 31, 2026",https://www.thegradcafe.com/result/1020288,Accepted,May 31,NaN,Fall 2026,International,NaN,NaN,PhD,NaN,NaN,Accepted on May 31 Fall 2026 International,"[Stockholm University, Environmental Economics...",1
2,Global Health PhD,Meharry Medical College,NaN,"May 30, 2026",https://www.thegradcafe.com/result/1020287,Accepted,May 29,NaN,Fall 2026,American,NaN,NaN,PhD,3.90,NaN,Accepted on May 29 Fall 2026 American GPA 3.90,"[Meharry Medical College, Global Health PhD, M...",1
3,Public Policy PhD,University of Massachusetts,NaN,"May 29, 2026",https://www.thegradcafe.com/result/1020286,Waitlisted,NaN,NaN,Fall 2026,International,NaN,NaN,PhD,NaN,NaN,Wait listed on May 29 Fall 2026 International,"[University of Massachusetts, Public Policy Ph...",1
4,Engineering Science PhD,University of Oxford,Applying as a (home student) Oxford undergrad....,"May 29, 2026",https://www.thegradcafe.com/result/1020285,Accepted,Apr 17,NaN,Fall 2026,Other,NaN,NaN,PhD,NaN,NaN,Accepted on Apr 17 Fall 2026 Other,"[University of Oxford, Engineering Science PhD...",1


In [28]:
df_cleaned.head(20)

,Program Name,University,Comments,Date of Information Added to Grad Cafe,URL link to applicant entry,Applicant Status,Accepted: Acceptance Date,Rejected: Rejection Date,Semester and Year of Program Start,International / American Student,GRE Score,GRE V Score,Masters or PhD,GPA,GRE AW,extra_info,raw_texts,page_number
0,Bioinformatics Masters,San Jose State University,NaN,"May 31, 2026",https://www.thegradcafe.com/result/1020289,Rejected,NaN,May 31,Fall 2026,American,NaN,NaN,Masters,3.20,NaN,Rejected on May 31 Fall 2026 American GPA 3.20,"[San Jose State University, Bioinformatics Mas...",1
1,Environmental Economics PhD,Stockholm University,NaN,"May 31, 2026",https://www.thegradcafe.com/result/1020288,Accepted,May 31,NaN,Fall 2026,International,NaN,NaN,PhD,NaN,NaN,Accepted on May 31 Fall 2026 International,"[Stockholm University, Environmental Economics...",1
2,Global Health PhD,Meharry Medical College,NaN,"May 30, 2026",https://www.thegradcafe.com/result/1020287,Accepted,May 29,NaN,Fall 2026,American,NaN,NaN,PhD,3.90,NaN,Accepted on May 29 Fall 2026 American GPA 3.90,"[Meharry Medical College, Global Health PhD, M...",1
3,Public Policy PhD,University of Massachusetts,NaN,"May 29, 2026",https://www.thegradcafe.com/result/1020286,Waitlisted,NaN,NaN,Fall 2026,International,NaN,NaN,PhD,NaN,NaN,Wait listed on May 29 Fall 2026 International,"[University of Massachusetts, Public Policy Ph...",1
4,Engineering Science PhD,University of Oxford,Applying as a (home student) Oxford undergrad....,"May 29, 2026",https://www.thegradcafe.com/result/1020285,Accepted,Apr 17,NaN,Fall 2026,Other,NaN,NaN,PhD,NaN,NaN,Accepted on Apr 17 Fall 2026 Other,"[University of Oxford, Engineering Science PhD...",1
5,Mechanical Engineering PhD,Rice University,"Duke Master of Science in ME, GPA: 3.85","May 29, 2026",https://www.thegradcafe.com/result/1020284,Accepted,May 08,NaN,Fall 2026,International,NaN,NaN,PhD,3.84,NaN,Accepted on May 08 Fall 2026 International GPA...,"[Rice University, Mechanical Engineering PhD, ...",1
6,Computer Science PhD,Georgia Institute of Technology,NaN,"May 29, 2026",https://www.thegradcafe.com/result/1020283,Rejected,NaN,May 29,Fall 2026,American,NaN,NaN,PhD,3.90,NaN,Rejected on May 29 Fall 2026 American GPA 3.90,"[Georgia Institute of Technology, Computer Sci...",1
7,Computer Science and Engineering PhD,University of Delaware,NaN,"May 29, 2026",https://www.thegradcafe.com/result/1020282,Waitlisted,NaN,NaN,Fall 2026,International,NaN,NaN,PhD,3.50,NaN,Wait listed on May 29 Fall 2026 International ...,"[University of Delaware, Computer Science and ...",1
8,Neuroscience Masters,McGill University,NaN,"May 29, 2026",https://www.thegradcafe.com/result/1020281,Accepted,May 25,NaN,Fall 2026,International,NaN,NaN,Masters,NaN,NaN,Accepted on May 25 Fall 2026 International,"[McGill University, Neuroscience Masters, May ...",1
9,Consumer Science Masters,Ohio State University,NaN,"May 29, 2026",https://www.thegradcafe.com/result/1020280,Accepted,May 15,NaN,Fall 2026,International,NaN,NaN,Masters,3.84,NaN,Accepted on May 15 Fall 2026 International GPA...,"[Ohio State University, Consumer Science Maste...",1
